In [22]:
import warnings
warnings.filterwarnings('ignore')

import gc
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

In [23]:
train_df=pd.read_parquet('/Users/rageshwer/Goal ML/Projects/Home_Credit_Default_Risk/data/interim/final_train.parquet')
test_df=pd.read_parquet('/Users/rageshwer/Goal ML/Projects/Home_Credit_Default_Risk/data/interim/final_test.parquet')

In [24]:
# ============================================================
# CONFIG
# ============================================================

TARGET_COL = 'TARGET'
ID_COL = 'SK_ID_CURR'

N_SPLITS = 5
RANDOM_STATE = 42

In [25]:
str_cols = train_df.select_dtypes(
    include=['object', 'string']
).columns

print(str_cols)

for col in str_cols:
    train_df[col] = train_df[col].astype('category')
    test_df[col] = test_df[col].astype('category')

Index(['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY'], dtype='str')


In [26]:
# ============================================================
# FEATURES
# ============================================================

features = [
    col for col in train_df.columns
    if col not in [TARGET_COL, ID_COL]
]

print(f"Number of Features: {len(features)}")

X = train_df[features]
y = train_df[TARGET_COL]

X_test = test_df[features]

Number of Features: 998


In [27]:
# ============================================================
# CATEGORICAL FEATURES
# ============================================================

cat_cols = X.select_dtypes(
    include=['category']
).columns.tolist()

print(f"Categorical Features: {len(cat_cols)}")

Categorical Features: 18


In [28]:
# ============================================================
# LIGHTGBM PARAMETERS
# ============================================================
params = {
    'objective': 'binary',
    'metric': 'auc',

    'boosting_type': 'gbdt',

    'learning_rate': 0.02,

    'n_estimators': 15000,

    'num_leaves': 128,

    'max_depth': -1,

    'min_child_samples': 50,

    'subsample': 0.9,
    'subsample_freq': 1,

    'colsample_bytree': 0.9,

    'reg_alpha': 0.1,
    'reg_lambda': 1.0,

    'random_state': 42,
    'n_jobs': -1,

    'verbosity': -1
}


In [29]:

# ============================================================
# CV SETUP
# ============================================================

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)


In [30]:

# ============================================================
# STORAGE
# ============================================================

oof_preds = np.zeros(len(train_df))

test_preds = np.zeros(len(test_df))

cv_scores = []

feature_importance = pd.DataFrame()

In [31]:
# ============================================================
# TRAINING LOOP
# ============================================================

for fold, (train_idx, valid_idx) in enumerate(
        skf.split(X, y), 1):

    print("=" * 60)
    print(f"FOLD {fold}")
    print("=" * 60)

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_valid = X.iloc[valid_idx]
    y_valid = y.iloc[valid_idx]

    model = lgb.LGBMClassifier(
        **params
    )

    model.fit(
        X_train,
        y_train,

        eval_set=[
            (X_valid, y_valid)
        ],

        eval_metric='auc',

        categorical_feature=cat_cols,

        callbacks=[
            lgb.early_stopping(
                stopping_rounds=300,
                verbose=True
            ),
            lgb.log_evaluation(
                period=200
            )
        ]
    )

    # ========================================================
    # VALIDATION PREDICTIONS
    # ========================================================

    valid_preds = model.predict_proba(
        X_valid,
        num_iteration=model.best_iteration_
    )[:, 1]

    oof_preds[valid_idx] = valid_preds

    # ========================================================
    # TEST PREDICTIONS
    # ========================================================

    fold_test_preds = model.predict_proba(
        X_test,
        num_iteration=model.best_iteration_
    )[:, 1]

    test_preds += fold_test_preds / N_SPLITS

    # ========================================================
    # FOLD SCORE
    # ========================================================

    fold_auc = roc_auc_score(
        y_valid,
        valid_preds
    )

    cv_scores.append(fold_auc)

    print(
        f"Fold {fold} AUC = {fold_auc:.6f}"
    )

    # ========================================================
    # FEATURE IMPORTANCE
    # ========================================================

    fold_importance = pd.DataFrame({
        'feature': features,
        'importance': model.feature_importances_,
        'fold': fold
    })

    feature_importance = pd.concat(
        [feature_importance, fold_importance],
        axis=0,
        ignore_index=True
    )

    del (
        X_train,
        X_valid,
        y_train,
        y_valid,
        valid_preds,
        fold_test_preds
    )

    gc.collect()

FOLD 1
Training until validation scores don't improve for 300 rounds
[200]	valid_0's auc: 0.767648
[400]	valid_0's auc: 0.772795
[600]	valid_0's auc: 0.773798
[800]	valid_0's auc: 0.774239
[1000]	valid_0's auc: 0.773979
Early stopping, best iteration is:
[820]	valid_0's auc: 0.774301
Fold 1 AUC = 0.774301
FOLD 2
Training until validation scores don't improve for 300 rounds
[200]	valid_0's auc: 0.775317
[400]	valid_0's auc: 0.781817
[600]	valid_0's auc: 0.782893
[800]	valid_0's auc: 0.782907
[1000]	valid_0's auc: 0.782358
Early stopping, best iteration is:
[879]	valid_0's auc: 0.783043
Fold 2 AUC = 0.783043
FOLD 3
Training until validation scores don't improve for 300 rounds
[200]	valid_0's auc: 0.768826
[400]	valid_0's auc: 0.774566
[600]	valid_0's auc: 0.775557
[800]	valid_0's auc: 0.775449
Early stopping, best iteration is:
[603]	valid_0's auc: 0.775638
Fold 3 AUC = 0.775638
FOLD 4
Training until validation scores don't improve for 300 rounds
[200]	valid_0's auc: 0.775365
[400]	valid

In [32]:
# ============================================================
# OVERALL CV SCORE
# ============================================================

overall_auc = roc_auc_score(
    y,
    oof_preds
)

print("\n")
print("=" * 60)
print("CROSS VALIDATION RESULTS")
print("=" * 60)

for i, score in enumerate(cv_scores, 1):
    print(f"Fold {i}: {score:.6f}")

print("\n")

print(
    f"Mean CV AUC : {np.mean(cv_scores):.6f}"
)

print(
    f"Std CV AUC  : {np.std(cv_scores):.6f}"
)

print(
    f"OOF AUC      : {overall_auc:.6f}"
)




CROSS VALIDATION RESULTS
Fold 1: 0.774301
Fold 2: 0.783043
Fold 3: 0.775638
Fold 4: 0.782889
Fold 5: 0.774047


Mean CV AUC : 0.777983
Std CV AUC  : 0.004104
OOF AUC      : 0.777818


In [33]:

# ============================================================
# FEATURE IMPORTANCE
# ============================================================

feature_importance_mean = (
    feature_importance
    .groupby('feature')['importance']
    .mean()
    .reset_index()
    .sort_values(
        by='importance',
        ascending=False
    )
)

print("\nTop 50 Features")
print(
    feature_importance_mean.head(50)
)


Top 50 Features
                                   feature  importance
943                      ORGANIZATION_TYPE      4747.2
972                      credit_to_annuity      2055.6
988                        goods_to_credit      1398.0
874                           EXT_SOURCE_3      1055.6
872                           EXT_SOURCE_1       992.8
860                 DAYS_LAST_PHONE_CHANGE       925.2
987                                ext_std       886.6
941                        OCCUPATION_TYPE       882.6
271                            AMT_ANNUITY       880.8
873                           EXT_SOURCE_2       831.4
859                        DAYS_ID_PUBLISH       815.4
968                      annuity_to_income       799.2
980                  ext_mean_x_employment       790.8
978                               ext_mean       767.0
990                   id_publish_age_ratio       763.6
857                             DAYS_BIRTH       737.4
861                      DAYS_REGISTRATION      

In [34]:


# ============================================================
# SAVE OOF
# ============================================================

oof_df = pd.DataFrame({
    ID_COL: train_df[ID_COL],
    TARGET_COL: y,
    'OOF_PRED': oof_preds
})

oof_df.to_parquet(
    'lgb_baseline_oof.parquet',
    index=False
)

print(
    "\nSaved: lgb_baseline_oof.parquet"
)

# ============================================================
# SAVE SUBMISSION
# ============================================================

submission = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    TARGET_COL: test_preds
})

submission.to_csv(
    'lgb_baseline_submission.csv',
    index=False
)

print(
    "Saved: lgb_baseline_submission.csv"
)

# ============================================================
# SAVE FEATURE IMPORTANCE
# ============================================================

feature_importance_mean.to_csv(
    'lgb_feature_importance.csv',
    index=False
)

print(
    "Saved: lgb_feature_importance.csv"
)


Saved: lgb_baseline_oof.parquet
Saved: lgb_baseline_submission.csv
Saved: lgb_feature_importance.csv
